# ParaTrace on Amazon SageMaker AI

ParaTrace tells whether an English text was written by a person or by AI. This notebook deploys the ParaTrace model
package from AWS Marketplace in your account and uses it in both ways SageMaker AI offers:

1. a **real-time endpoint** on ml.g4dn.xlarge (one NVIDIA T4), asked with JSON, plain text and JSON Lines;
2. a **batch transform job**, which checks a file of texts in Amazon S3.

It brings its own sample texts, and deletes the endpoint, the job's model and its files at the end. Running it takes
about 20 minutes and costs ml.g4dn.xlarge's price for that time, in SageMaker AI and for ParaTrace's subscription.

**Before you start**

* Subscribe to ParaTrace on AWS Marketplace.
* Run the notebook in SageMaker AI Studio or on a notebook instance whose role may create SageMaker AI models,
  endpoints and transform jobs, and use SageMaker AI's default S3 bucket (for example with the managed policy
  AmazonSageMakerFullAccess).
* Your account needs the SageMaker AI quotas *ml.g4dn.xlarge for endpoint usage* and *ml.g4dn.xlarge for transform
  job usage* of at least 1 (Service Quotas).

## 1. Set up

ParaTrace needs a recent boto3 for the settings that pick SageMaker AI's newer images. If boto3 was already imported in
this kernel, restart the kernel after this cell.

In [ ]:
%pip install --quiet --upgrade boto3

In [ ]:
import json
import os
import time

import boto3
from botocore.exceptions import ClientError

# The ParaTrace model package of your subscription in each AWS Region (AWS Marketplace shows it once you have
# subscribed: Configure, then the Region).
MODEL_PACKAGES = {
}

INSTANCE_TYPE = "ml.g4dn.xlarge"                            # one NVIDIA T4
# SageMaker AI's default image of ml.g4dn has NVIDIA driver 470, too old for ParaTrace: these are newer ones.
INFERENCE_AMI = "al2023-ami-sagemaker-inference-gpu-4-1"    # real-time endpoints: driver 580
TRANSFORM_AMI = "al2-ami-sagemaker-batch-gpu-535"           # batch transform: driver 535

session = boto3.Session()
region = session.region_name
account = session.client("sts").get_caller_identity()["Account"]
sm = session.client("sagemaker")             # SageMaker AI
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")

model_package = os.environ.get("PARATRACE_MODEL_PACKAGE") or MODEL_PACKAGES.get(region)
if not model_package:
    raise ValueError(f"No ParaTrace model package for {region}: run the notebook in a Region of MODEL_PACKAGES")
print(f"Region {region}, model package {model_package}")

In [ ]:
def execution_role():
    """The IAM role this notebook runs with in SageMaker AI, which the model uses too."""
    try:
        from sagemaker import get_execution_role            # the SageMaker Python SDK of SageMaker AI's images
        return get_execution_role()
    except Exception:                                       # another version of it, or none: ask IAM
        arn = session.client("sts").get_caller_identity()["Arn"]   # ...:assumed-role/<role>/<session>
        if ":assumed-role/" not in arn:
            raise ValueError(f"Run the notebook in SageMaker AI, or set ROLE to a role it can assume (not {arn})")
        name = arn.split(":assumed-role/")[1].split("/")[0]
        return session.client("iam").get_role(RoleName=name)["Role"]["Arn"]


ROLE = execution_role()

# SageMaker AI's default bucket, made if it isn't there yet: the batch transform's input and output go into it.
BUCKET = f"sagemaker-{region}-{account}"
try:
    s3.head_bucket(Bucket=BUCKET)
except ClientError:
    s3.create_bucket(Bucket=BUCKET, **({} if region == "us-east-1" else
                                       {"CreateBucketConfiguration": {"LocationConstraint": region}}))

NAME = "paratrace-" + time.strftime("%Y%m%d-%H%M%S")      # the model's, the endpoint's and the job's
print(f"Role {ROLE}\nBucket {BUCKET}\nName {NAME}")

Four sample texts: two written by people, in the public domain, and two written by an AI assistant for this notebook.

In [ ]:
GETTYSBURG = "Four score and seven years ago our fathers brought forth on this continent, a new nation, conceived in Liberty, and dedicated to the proposition that all men are created equal.\n\nNow we are engaged in a great civil war, testing whether that nation, or any nation so conceived and so dedicated, can long endure. We are met on a great battle-field of that war. We have come to dedicate a portion of that field, as a final resting place for those who here gave their lives that that nation might live. It is altogether fitting and proper that we should do this.\n\nBut, in a larger sense, we can not dedicate -- we can not consecrate -- we can not hallow -- this ground. The brave men, living and dead, who struggled here, have consecrated it, far above our poor power to add or detract. The world will little note, nor long remember what we say here, but it can never forget what they did here. It is for us the living, rather, to be dedicated here to the unfinished work which they who fought here have thus far so nobly advanced. It is rather for us to be here dedicated to the great task remaining before us -- that from these honored dead we take increased devotion to that cause for which they gave the last full measure of devotion -- that we here highly resolve that these dead shall not have died in vain -- that this nation, under God, shall have a new birth of freedom -- and that government of the people, by the people, for the people, shall not perish from the earth."

PRIDE_AND_PREJUDICE = "It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife.\n\nHowever little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered the rightful property of some one or other of their daughters.\n\n\"My dear Mr. Bennet,\" said his lady to him one day, \"have you heard that Netherfield Park is let at last?\"\n\nMr. Bennet replied that he had not.\n\n\"But it is,\" returned she; \"for Mrs. Long has just been here, and she told me all about it.\"\n\nMr. Bennet made no answer.\n\n\"Do you not want to know who has taken it?\" cried his wife impatiently.\n\n\"You want to tell me, and I have no objection to hearing it.\"\n\nThis was invitation enough."

REMOTE_WORK = "Remote work has fundamentally reshaped how organizations approach productivity and collaboration. By eliminating daily commutes, employees can reclaim valuable time and often report a better work-life balance. At the same time, companies gain access to a broader talent pool that is no longer constrained by geography.\n\nHowever, remote work also presents unique challenges. Without the spontaneous interactions of a shared office, teams may struggle to maintain a strong sense of connection and culture. Clear communication becomes essential, and leaders must be intentional about fostering trust and inclusion.\n\nTo make remote work successful, organizations should consider the following strategies:\n\n- Establish clear expectations around availability and response times.\n- Invest in reliable collaboration tools and digital infrastructure.\n- Schedule regular check-ins to support both performance and well-being.\n- Create opportunities for informal social connection.\n\nUltimately, the most effective approach is often a thoughtful hybrid model that combines the flexibility of remote work with the benefits of in-person collaboration. By prioritizing communication, trust, and employee well-being, organizations can build resilient teams that thrive in any environment."

PHOTOSYNTHESIS = "Photosynthesis is the remarkable process by which plants, algae, and certain bacteria convert light energy into chemical energy. During this process, chlorophyll in the chloroplasts absorbs sunlight, which powers the transformation of carbon dioxide and water into glucose and oxygen. This not only provides the energy that fuels plant growth but also releases the oxygen that sustains most life on Earth. In essence, photosynthesis serves as the foundation of nearly every food chain, making it one of the most important biological processes on our planet."

## 2. The model

A SageMaker AI model of the model package. AWS Marketplace models run with network isolation: no text leaves the
container.

In [ ]:
sm.create_model(ModelName=NAME, ExecutionRoleArn=ROLE, EnableNetworkIsolation=True,
                       PrimaryContainer={"ModelPackageName": model_package})

## 3. A real-time endpoint

On ml.g4dn.xlarge with the newer SageMaker AI image (`InferenceAmiVersion`). It takes about 5 minutes to start.

In [ ]:
sm.create_endpoint_config(EndpointConfigName=NAME, ProductionVariants=[{
    "VariantName": "all", "ModelName": NAME, "InstanceType": INSTANCE_TYPE, "InitialInstanceCount": 1,
    "InferenceAmiVersion": INFERENCE_AMI}])
sm.create_endpoint(EndpointName=NAME, EndpointConfigName=NAME)
print("The endpoint is starting ...")
sm.get_waiter("endpoint_in_service").wait(EndpointName=NAME)
print("In service")

In [ ]:
def check(body, content_type="application/json"):
    """ParaTrace's answer to one request: a dict, or for JSON Lines a list with one dict per line."""
    if not isinstance(body, (str, bytes)):
        body = json.dumps(body)
    answer = runtime.invoke_endpoint(EndpointName=NAME, ContentType=content_type, Body=body)
    text = answer["Body"].read().decode("utf-8")
    if content_type == "application/jsonlines":
        return [json.loads(line) for line in text.splitlines()]
    return json.loads(text)


def show(verdicts):
    """One line per verdict."""
    for v in verdicts:
        if "error" in v:
            print(f"{v['document']:24} error: {v['error']}")
        else:
            print(f"{v['document']:24} {v['verdict']:6} P(AI) {v['probability']:.4f}   confidence {v['confidence']:6}"
                  f"   {v['operating_point']}, {v['n_words']} words {v['warnings'] or ''}")

### JSON

Up to 10 texts per request in `texts` (or one in `text`), each up to 200,000 characters. `operating_point` is `strict`
(the default, about 1 in 100 human texts flagged) or `lenient` (about 5 in 100).

In [ ]:
answer = check({"texts": [GETTYSBURG, REMOTE_WORK], "operating_point": "strict"})
print("ParaTrace", answer["detector_version"])
show(answer["documents"])

Detailed answers (the default `mode`) also list each 512-token window's probability and the text's **segments**: runs
of sentences labelled `ai` or `human`, with their offsets in the text as sent.

In [ ]:
document = answer["documents"][1]
for segment in document["segments"]:
    passage = REMOTE_WORK[segment["start"]:segment["end"]]
    print(f"{segment['label']:5} P(AI) {segment['probability']:.4f}  {passage[:90]!r}...")

`"mode": "fast"` answers the verdicts only, without windows and segments.

In [ ]:
answer = check({"text": PRIDE_AND_PREJUDICE, "mode": "fast"})
show(answer["documents"])
print(sorted(answer["documents"][0]))

### Plain text

One text as `text/plain`, answered as JSON.

In [ ]:
show(check(PHOTOSYNTHESIS, "text/plain")["documents"])

### JSON Lines

One text per line, as a JSON string or as an object with `text` and, optionally, `document` (a name), `operating_point`
and `mode`. Every line gets an answer line in the same order; a line that can't be checked gets an `error`.

In [ ]:
lines = [
    json.dumps({"document": "gettysburg-address", "text": GETTYSBURG}),
    json.dumps({"document": "remote-work-answer", "text": REMOTE_WORK, "operating_point": "lenient"}),
    json.dumps({"document": "pride-and-prejudice", "text": PRIDE_AND_PREJUDICE, "mode": "fast"}),
    json.dumps(PHOTOSYNTHESIS),
    json.dumps({"document": "empty", "text": "   "}),
]
show(check("\n".join(lines), "application/jsonlines"))

The endpoint is billed for every hour it runs: delete it once you no longer need it.

In [ ]:
sm.delete_endpoint(EndpointName=NAME)
sm.delete_endpoint_config(EndpointConfigName=NAME)

## 4. A batch transform job

For many texts: a JSON Lines file in S3, one text per line, answered line by line in `<file>.out`. The job runs on
ml.g4dn.xlarge with the newer SageMaker AI image (`TransformAmiVersion`) and takes about 10 minutes for this small
file, most of it starting the instance.

In [ ]:
prefix = f"paratrace/{NAME}"
s3.put_object(Bucket=BUCKET, Key=f"{prefix}/input/texts.jsonl", Body="\n".join(lines[:4]) + "\n")

sm.create_transform_job(
    TransformJobName=NAME, ModelName=NAME,
    BatchStrategy="MultiRecord", MaxPayloadInMB=4, MaxConcurrentTransforms=1,
    TransformInput={"DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BUCKET}/{prefix}/input/"}},
                    "ContentType": "application/jsonlines", "SplitType": "Line"},
    TransformOutput={"S3OutputPath": f"s3://{BUCKET}/{prefix}/output/", "Accept": "application/jsonlines",
                     "AssembleWith": "Line"},
    TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1, "TransformAmiVersion": TRANSFORM_AMI})
print("The batch transform job is running ...")
sm.get_waiter("transform_job_completed_or_stopped").wait(TransformJobName=NAME)
job = sm.describe_transform_job(TransformJobName=NAME)
print(job["TransformJobStatus"], job.get("FailureReason", ""))

In [ ]:
output = s3.get_object(Bucket=BUCKET, Key=f"{prefix}/output/texts.jsonl.out")["Body"].read().decode("utf-8")
show([json.loads(line) for line in output.splitlines()])

## 5. Clean up

The model, and the job's input and output in S3.

In [ ]:
sm.delete_model(ModelName=NAME)
for item in s3.list_objects_v2(Bucket=BUCKET, Prefix=prefix + "/").get("Contents", []):
    s3.delete_object(Bucket=BUCKET, Key=item["Key"])
print("Deleted")

## What the answers mean

| Field | |
|---|---|
| `verdict` | `ai`, `human` or `mixed`: `ai` when at least 80% of the text's characters are labelled AI, `human` when at least 90% are labelled human |
| `probability` | P(AI-generated), 0 to 1: it ranks texts, the verdict decides them |
| `confidence` | `low`, `medium` or `high`: how far the evidence lies from the threshold |
| `ai_fraction` | The share of the characters labelled AI |
| `warnings` | For example `short text: verdict unreliable` under 50 words |
| `windows`, `segments` | Detailed answers only: offsets in Unicode code points of the text as sent, end exclusive |

ParaTrace checks English texts, and its verdicts are estimates, not proof of who wrote a text. The repository's README
has every field, the speed on ml.g4dn.xlarge, the accuracy and the known limitations.